# 📚 Educational Content Agent Pipeline
**A Dual-Agent Architecture for Automated Content Generation and Review**

This notebook implements an AI-driven pipeline to generate structured educational content. It utilizes a two-agent system—a Generator and a Reviewer—to ensure the output is age-appropriate, conceptually accurate, and clearly written.

In [2]:
# Install dependencies silently and import required libraries
!pip install -q google-genai pydantic gradio

import os
import json
import gradio as gr
from pydantic import BaseModel
from typing import List
from google import genai
from google.genai import types

## 1. Data Structures & Schemas
To ensure our AI agents produce deterministic and reliable outputs, we enforce strict data schemas as mentioned in the assessment guide provided.

In [3]:
class MCQ(BaseModel):
    question: str
    options: List[str]
    answer: str

class GeneratorSchema(BaseModel):
    explanation: str
    mcqs: List[MCQ]

class ReviewerSchema(BaseModel):
    status: str
    feedback: List[str]

## 2. Core AI Agents
This section defines the specialized roles within our pipeline. We utilize the Gemini 2.5 Flash model for its speed and native support for structured JSON generation.

* **The Generator Agent:** Responsible for drafting initial educational explanations and multiple-choice questions based on a target grade and topic.
* **The Reviewer Agent:** Acts as an automated quality-assurance gate, evaluating the drafted content against strict criteria (age-appropriateness, clarity, and accuracy) and returning actionable feedback.

In [4]:
class GeneratorAgent:
    def __init__(self, model="gemini-2.5-flash"):
        self.client = genai.Client()
        self.model = model

    def generate(self, grade, topic, feedback=None):
        prompt = f"Create educational content for grade {grade} students about the topic: '{topic}'."
        if feedback:
            prompt += f"\nRefine your previous output by fixing these specific issues: {feedback}"

        try:
            response = self.client.models.generate_content(
                model=self.model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction="You are an expert educational content creator.",
                    response_mime_type="application/json",
                    response_json_schema=GeneratorSchema.model_json_schema(),
                    temperature=0.4,
                ),
            )
            return json.loads(response.text)
        except Exception as e:
            return {"error": f"Generator API Error: {str(e)}"}

class ReviewerAgent:
    def __init__(self, model="gemini-2.5-flash"):
        self.client = genai.Client()
        self.model = model

    def review(self, generator_content):
        prompt = f"""Evaluate the following educational content:
        {json.dumps(generator_content, indent=2)}

        Evaluate strictly based on: 1. Age appropriateness 2. Conceptual correctness 3. Clarity.
        Status must be strictly 'pass' or 'fail'. If pass, leave feedback empty.
        """
        try:
            response = self.client.models.generate_content(
                model=self.model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction="You are a strict educational content reviewer.",
                    response_mime_type="application/json",
                    response_json_schema=ReviewerSchema.model_json_schema(),
                    temperature=0.1,
                ),
            )
            return json.loads(response.text)
        except Exception as e:
            return {"status": "fail", "feedback": [f"Reviewer API Error: {str(e)}"]}

## 3. Pipeline Orchestration & Refinement Logic
With our agents defined, we now orchestrate their interaction.

The pipeline function below manages the state between the Generator and the Reviewer. If the initial draft fails the Reviewer's evaluation, the pipeline automatically captures the specific feedback and triggers a single inline refinement pass, allowing the Generator to correct its mistakes before returning the final payload.

In [5]:
def run_agent_pipeline(api_key, grade, topic):
    if not api_key:
        return '{"error": "Please enter your Gemini API Key."}', "", ""

    os.environ["GEMINI_API_KEY"] = api_key
    generator = GeneratorAgent()
    reviewer = ReviewerAgent()

    draft_content = generator.generate(grade, topic)
    if "error" in draft_content:
        return json.dumps(draft_content, indent=2), "", ""

    review_result = reviewer.review(draft_content)

    refined_content = None
    if review_result.get("status").lower() == "fail" and review_result.get("feedback"):
        refined_content = generator.generate(grade, topic, feedback=review_result.get("feedback"))

    draft_str = json.dumps(draft_content, indent=2)
    review_str = json.dumps(review_result, indent=2)
    refined_str = json.dumps(refined_content, indent=2) if refined_content else "✅ Draft passed initial review. No refinement needed."

    return draft_str, review_str, refined_str

## 4. Interactive User Interface
To make this agentic workflow accessible and easy to test, we wrap the pipeline in a web-based interface.

Using Gradio, we expose the input parameters (API Key, Grade, Topic) and visually map the pipeline's execution flow. This allows users to clearly see the initial draft, the reviewer's evaluation, and the finalized, refined content in real-time.

In [6]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📚 Educational Content Agent Pipeline")
    gr.Markdown("A UI-driven, two-agent architecture for generating and reviewing educational content.")

    with gr.Row():
        with gr.Column(scale=1):
            api_key = gr.Textbox(label="Gemini API Key", type="password", placeholder="Enter your key here...")
            grade = gr.Number(label="Target Grade", value=4, precision=0)
            topic = gr.Textbox(label="Topic", value="Types of angles")
            submit_btn = gr.Button("Generate Content", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### 1. Initial Generator Draft")
            out_draft = gr.Code(language="json", interactive=False)

            gr.Markdown("### 2. Reviewer Evaluation")
            out_review = gr.Code(language="json", interactive=False)

            gr.Markdown("### 3. Refined Output")
            out_refined = gr.Code(language="json", interactive=False)

    submit_btn.click(
        fn=run_agent_pipeline,
        inputs=[api_key, grade, topic],
        outputs=[out_draft, out_review, out_refined]
    )

demo.launch(debug=True)

/tmp/ipykernel_2972/3204924670.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5f86a082c95f6d1e93.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5f86a082c95f6d1e93.gradio.live
